# 📦 Ejercicios: FedEx
# Envíos - servicio, peso, retrasos y coste

⚠️ Datos SIMULADOS / ilustrativos
(generados con pandas + numpy).
No son cifras oficiales de FedEx.

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué servicio llega
### tarde más veces?
### (tasa de retraso)

La puntualidad es LA promesa de
FedEx. Un retraso rompe la confianza.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_envios.csv'
)

r = (
  df
  .groupby('servicio')
  .agg(
    retraso_pct=('retrasado',
      lambda s: s.mean() * 100),
    n=('envio_id', 'count'),
  )
  .round(1)
  .sort_values('retraso_pct',
    ascending=False)
)

print('Retraso por servicio:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿El peso dispara
### el retraso? (tramos
### de peso)

Troceamos el peso en tramos y vemos
si los paquetes pesados fallan más.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_envios.csv'
)

df['tramo_peso'] = pd.cut(
  df['peso_kg'],
  bins=[0, 2, 10, 30, 100],
  labels=['<2kg', '2-10', '10-30',
          '>30'],
)

r = (
  df
  .groupby('tramo_peso',
    observed=True)['retrasado']
  .mean()
  .mul(100)
  .round(1)
)

print('Retraso % por peso:')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué ruta origen-
### destino es la más
### cara por kg?

Coste / kg revela rutas poco
rentables que hay que optimizar.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_envios.csv'
)

df['ruta'] = (
  df['hub_origen'] + '→'
  + df['hub_destino']
)
df['coste_kg'] = (
  df['coste_usd'] / df['peso_kg']
)

r = (
  df
  .groupby('ruta')['coste_kg']
  .mean()
  .round(2)
  .sort_values(ascending=False)
  .head(5)
)

print('Top rutas caras ($/kg):')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Cuántos días de más
### tardamos cuando
### fallamos? (retraso medio)

Días reales - prometidos mide la
gravedad del fallo, no solo si pasó.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_envios.csv'
)

df['dias_extra'] = (
  df['dias_reales']
  - df['dias_prometidos']
)

tarde = df[df['dias_extra'] > 0]

r = (
  tarde
  .groupby('zona')['dias_extra']
  .mean()
  .round(2)
  .sort_values(ascending=False)
)

print('Días extra medios (tarde):')
print(r)

### 💼 PREGUNTA ENTREVISTA:
### ¿Qué hub de salida
### es el más fiable
### y barato a la vez?

Cruzamos puntualidad y coste medio
por hub para evaluar la red.

In [ ]:
import pandas as pd

df = pd.read_csv(
  '../datasets/fedex_envios.csv'
)

r = (
  df
  .groupby('hub_origen')
  .agg(
    retraso=('retrasado',
      lambda s: s.mean() * 100),
    coste=('coste_usd', 'mean'),
  )
  .round(2)
  .sort_values('retraso')
)

print('Fiabilidad y coste por hub:')
print(r)